# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [3]:
# import numpy and pandas
import pandas as pd
import numpy as np
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta
from statsmodels.stats.weightstats import ztest as ztest

from scipy.stats import ttest_ind, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem

# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [74]:
# Run this code:
salaries = pd.read_csv('../data/Current_Employee_Names__Salaries__and_Position_Titles.csv')

Examine the `salaries` dataset using the `head` function below.

In [77]:
# Your code here
salaries.head()
salaries.info()

<class 'pandas.DataFrame'>
RangeIndex: 33183 entries, 0 to 33182
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               33183 non-null  str    
 1   Job Titles         33183 non-null  str    
 2   Department         33183 non-null  str    
 3   Full or Part-Time  33183 non-null  str    
 4   Salary or Hourly   33183 non-null  str    
 5   Typical Hours      8022 non-null   float64
 6   Annual Salary      25161 non-null  float64
 7   Hourly Rate        8022 non-null   float64
dtypes: float64(3), str(5)
memory usage: 3.6 MB


# Challenge 2 - Hypothesis Tests

In this section of the lab, we will test whether the hourly wage of all hourly workers is significantly different from $30/hr. Import the correct one sample test function from scipy and perform the hypothesis test for a 95% two sided confidence interval.

In [84]:

import scipy.stats as stats
#rng = np.random.default_rng(42)

# H0 -> wage  $30/h
h0=30
# select ll hourly workers wages:
salaries_per_hour =salaries["Hourly Rate"].dropna()
#salaries_per_hour.info()
mu, sigma = salaries_per_hour.mean(), salaries_per_hour.std(ddof=0)

# get a sample of the data
#sample_salaries = rng.choice(salaries_per_hour, size=400, replace=False)

#compute mean and std_err of the mean
#x_bar, se = sample_salaries.mean(), stats.sem(sample_salaries)
#print (x_bar, se)

print(f"true μ  : {mu:.2f}")

result = stats.ttest_1samp(salaries_per_hour, popmean=h0)
print("test statistic=", result.statistic, "pvalue=",result.pvalue)
ci_95= result.confidence_interval(confidence_level=0.95)
print(f"95% CI: ({ci_95.low:.4f}, {ci_95.high:.4f})")
print("The nul hypothesis of 30 is not inside the confidence interval, and p=value is very small, so it is rejected")

true μ  : 32.79
test statistic= 20.6198057854942 pvalue= 4.323024048623004e-92
95% CI: (32.5235, 33.0537)
The nul hypothesis of 30 is not inside the confidence interval, and p=value is very small, so it is rejected


# Challenge 3 - Constructing Confidence Intervals

While testing our hypothesis is a great way to gather empirical evidence for accepting or rejecting the hypothesis, another way to gather evidence is by creating a confidence interval. A confidence interval gives us information about the true mean of the population. So for a 95% confidence interval, we are 95% sure that the mean of the population is within the confidence interval. 
).

To read more about confidence intervals, click [here](https://en.wikipedia.org/wiki/Confidence_interval).


In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. 

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter and the standard error for the scale. The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [45]:
# Your code here
# compute mean and std_err of the mean of the sample
x_bar, se = sample_salaries.mean(), stats.sem(sample_salaries)
print (x_bar, se)
ci_95_low, ci_95_high = stats.t.interval( 0.95, df=len(sample_salaries)-1, loc=x_bar, scale=se)
print ( f"ci=[{ci_95_low}, {ci_95_high}]")


32.876000000000005 0.5897171266579293
ci=[31.71665900383115, 34.03534099616886]


# Challenge 4 - Hypothesis Tests of Proportions

Another type of one sample test is a hypothesis test of proportions. In this test, we examine whether the proportion of a group in our sample is significantly different than a fraction. 

You can read more about one sample proportion tests [here](http://sphweb.bumc.bu.edu/otlt/MPH-Modules/BS/SAS/SAS6-CategoricalData/SAS6-CategoricalData2.html).

In the cell below, use the `proportions_ztest` function from `statsmodels` to perform a hypothesis test that will determine whether the number of hourly workers in the City of Chicago is significantly different from 25% at the 95% confidence level.

In [73]:
# Your code here
# 

rng = np.random.default_rng(42)
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

nobs = 1000 # nr of observations
payment_type =salaries["Salary or Hourly"].dropna()
sample_payment= rng.choice(payment_type, size=nobs, replace=False)

sample_hourly = sample_payment[sample_payment == "Hourly"]

counts=len(sample_hourly)
print (f" {counts} workers from {nobs} are paid hourly")

zstat, pval = proportions_ztest(count=counts, nobs=nobs, value=0.25, alternative="two-sided")
print(f"z = {zstat:.4f}, p = {pval:.4f}")

ci_low, ci_high = proportion_confint( count=counts, nobs=nobs, alpha=0.05, method="normal")
 
print ( f"confidence interval 95%: ci=[{ci_low}, {ci_high}]")
counts

print("The number of hourly workers is not significantly different from the 95% confidence interval" )


 238 workers from 1000 are paid hourly
z = -0.8911, p = 0.3729
confidence interval 95%: ci=[0.21160546257488486, 0.26439453742511515]
The number of hourly workers is not significantly different from the 95% confidence interval
